# 🧩 `georisk_twin` v0.5 · Piloto de la librería y API para operacionalizar un Gemelo Digital Multimodal Urbano-Industrial

**Notebook piloto (prueba de concepto) en Google Colab**

**Tema de tesis:** *Librería y API en Python para operacionalizar la metodología de un gemelo digital multimodal urbano-industrial*
**Autores:** Zambrano Mendoza Jeremy Dario · Chango Burgos Michael Alexander
**Institución:** Universidad de Guayaquil · Facultad de Ingeniería Industrial · Sistemas de la Información

---

## ¿Qué hace este notebook?

A diferencia de un notebook de análisis, **aquí se construye el artefacto de software**: un paquete Python instalable llamado `georisk_twin`, con su API, pruebas, documentación, demo y plantilla de contribución. Todo se genera desde las celdas y queda listo para descargar como `.zip` y subir a GitHub.

> **Problema que resuelve (del anteproyecto):** ocho subproductos de tesis (SI-01, SI-02, IND-SIS-01, URB-01 a URB-05) se desarrollan en paralelo **sin una estructura de software común**. `georisk_twin` es esa estructura: define los *contratos de datos*, expone una API y permite conectar cada módulo sin reemplazarlo.

### Mapa de la metodología (Design Science Research) sobre este notebook

| Fase DSR | Qué se hace | Pasos |
|---|---|---|
| Identificación del problema | Se declara el problema y qué se va a construir | 1 |
| Definición de objetivos de solución | Se fijan los **4 criterios de aceptación** de la v0.5 | 1 |
| Diseño y desarrollo | Paquete, contratos Pydantic, módulos, endpoints FastAPI | 2 a 5 |
| Demostración | Instalación limpia y demo reproducible | 6 y 9 |
| Evaluación | pytest, cobertura, tiempos de respuesta, veredicto | 7, 8 y 12 |
| Comunicación | Documentación, plantilla de contribución, `.zip` entregable | 10, 11 y 13 |

### Prácticas del anteproyecto que se reflejan en el código

| Práctica | Cómo aparece |
|---|---|
| **API First Design** | Primero se escriben los *contratos* (Paso 3) y la forma de los endpoints; después la lógica |
| **OpenAPI** | FastAPI genera la especificación a partir de las rutas y modelos (Paso 8) |
| **Mocks y contratos de datos** (Fase 4 Kanban) | Los módulos de otras tesis aún no existen, así que se simulan respetando el mismo contrato (Pasos 4 y 10) |
| **Integración continua** | Se genera el flujo de GitHub Actions y se ejecuta localmente lo mismo que correría allí (Pasos 7 y 11) |
| **Historias de usuario HU-01…HU-06** | Se verifican una por una con evidencia automática (Paso 12) |

> ⚠️ **Aviso:** los módulos de datos e indicadores usan **datos simulados** (*mocks*) que respetan el contrato esperado de SI-02 y SI-01. El módulo analítico (IA) es una implementación real mínima con `scikit-learn`. Los módulos de grafos, simulación y visualización aparecen como **pendientes**, tal como indica el alcance de la v0.5 (umbral mínimo: datos + indicadores + un módulo analítico).

### Cómo usarlo
**Entorno de ejecución → Ejecutar todo** (≈ 2 a 3 minutos; la parte más lenta es la instalación y la compilación del paquete).

## Paso 0 · Preparación del entorno

Solo se necesitan herramientas de desarrollo; el propio paquete declarará sus dependencias en `pyproject.toml`.

In [ ]:
!pip install -q fastapi "uvicorn[standard]" pydantic pandas scikit-learn numpy httpx requests pytest pytest-cov build matplotlib

In [ ]:
import json, os, sys, time, shutil, subprocess, threading, textwrap, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

pd.set_option("display.width", 150); pd.set_option("display.max_columns", 30)

REPO = Path("georisk_twin_repo").resolve()      # carpeta donde se construirá el proyecto
PKG  = REPO / "src" / "georisk_twin"

def ejecutar(cmd, cwd=None, env=None, mostrar=True):
    """Ejecuta un comando y devuelve (código, salida). Sirve para pip, pytest, demos…"""
    p = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True)
    salida = (p.stdout + p.stderr).strip()
    if mostrar and salida:
        print(salida)
    return p.returncode, salida

print("Proyecto se creará en:", REPO)

## Paso 0B · (Opcional) Conexión a Supabase: cargar y guardar datos

Esta sección permite **leer** los datos de entrada desde tu proyecto de Supabase (PostgreSQL) y **guardar** los resultados del notebook en él. Viene **desactivada** (`USAR_SUPABASE = False`) para que *Ejecutar todo* funcione sin credenciales.

**Cómo activarla**
1. En Colab, abre el panel 🔑 **Secrets** (izquierda) y crea `SUPABASE_URL` y `SUPABASE_KEY` (Supabase → *Project Settings → API*). Activa «Acceso desde el cuaderno».
2. Usa la clave **anon / public** con **Row Level Security (RLS)** activado. **Nunca** pegues la clave `service_role` en un notebook: da control total sobre la base y quien reciba el archivo podría usarla.
3. Crea las tablas (SQL de abajo, en el *SQL Editor* de Supabase) y cambia `USAR_SUPABASE = True`.

Se incluyen los dos métodos de tu celda original: **A** (solicitudes `requests`, el que usa el notebook) y **B** (SDK oficial). Además, la carga pagina de a 1 000 filas (el máximo por defecto de Supabase) y el guardado admite **upsert** para poder re-ejecutar sin duplicar filas.


### SQL para este notebook
```sql
-- Incidentes de entrada (contrato `Incidente` de georisk_twin)
create table if not exists accidentes_georisk (
  id_evento text primary key, fecha date, zona text, tipo_evento text, severidad int, dias_perdidos int default 0);

-- Resultados del flujo datos → indicadores → analítico (una fila por zona)
create table if not exists georisk_resultados (
  zona text primary key, n_incidentes int, indice_frecuencia float8, indice_gravedad float8, exposicion_horas float8,
  puntaje_riesgo float8, nivel text, cluster int, prioridad text, version text);

-- Políticas de PRUEBA para un piloto (permisivas). Ajústalas antes de usar datos reales.
alter table accidentes_georisk enable row level security;
create policy "accidentes_georisk_lectura"   on accidentes_georisk for select to anon using (true);
create policy "accidentes_georisk_insercion" on accidentes_georisk for insert to anon with check (true);
create policy "accidentes_georisk_update"    on accidentes_georisk for update to anon using (true) with check (true);

-- Políticas de PRUEBA para un piloto (permisivas). Ajústalas antes de usar datos reales.
alter table georisk_resultados enable row level security;
create policy "georisk_resultados_lectura"   on georisk_resultados for select to anon using (true);
create policy "georisk_resultados_insercion" on georisk_resultados for insert to anon with check (true);
create policy "georisk_resultados_update"    on georisk_resultados for update to anon using (true) with check (true);
```

In [ ]:
# Instalar el cliente oficial de Supabase (opcional: el MÉTODO A usa solo `requests`)
!pip install -q supabase

import os, json, requests
import pandas as pd

# ───────────── 1. Interruptor y credenciales ─────────────
USAR_SUPABASE = False     # ← cámbialo a True cuando ya tengas tus credenciales reales

def _secreto(nombre, defecto=""):
    """Busca la credencial en los Secrets de Colab (🔑 panel izquierdo) y, si no, en variables de entorno."""
    try:
        from google.colab import userdata
        valor = userdata.get(nombre)
    except Exception:
        valor = None
    return valor or os.environ.get(nombre, defecto)

# Project Settings → API (en tu panel de Supabase). Lo recomendable es guardarlas como Secrets.
SUPABASE_URL = _secreto("SUPABASE_URL", "https://xxxxxxxx.supabase.co")
SUPABASE_KEY = _secreto("SUPABASE_KEY", "")          # usa la clave «anon/public», NUNCA la «service_role»

def _supabase_listo():
    if not USAR_SUPABASE:
        print("ℹ️ Supabase desactivado (USAR_SUPABASE = False).")
        return False
    if "xxxxxxxx" in SUPABASE_URL or not SUPABASE_KEY:
        print("⚠️ Faltan credenciales: define SUPABASE_URL y SUPABASE_KEY (Secrets de Colab).")
        return False
    return True

def _headers(extra=None):
    h = {"apikey": SUPABASE_KEY, "Authorization": f"Bearer {SUPABASE_KEY}", "Content-Type": "application/json"}
    if extra:
        h.update(extra)
    return h

# ───────────── 2. CARGAR datos ─────────────
# MÉTODO A: solicitudes HTTP básicas (requests). Pagina de a 1000 filas (límite por defecto de Supabase).
def cargar_datos_api_rest(tabla, columnas="*", filtros=None, pagina=1000):
    """Equivale a SELECT <columnas> FROM <tabla>. `filtros` p. ej. {"zona": "eq.Z01"}."""
    if not _supabase_listo():
        return None
    print(f"📡 Consultando API REST de Supabase → {tabla} ...")
    filas, inicio = [], 0
    while True:
        r = requests.get(f"{SUPABASE_URL}/rest/v1/{tabla}", params={"select": columnas, **(filtros or {})},
                         headers=_headers({"Range-Unit": "items", "Range": f"{inicio}-{inicio + pagina - 1}"}), timeout=30)
        if r.status_code not in (200, 206):
            print(f"❌ Error en la llamada API: {r.status_code}\n{r.text}")
            return None
        lote = r.json()
        filas += lote
        if len(lote) < pagina:
            break
        inicio += pagina
    print(f"   {len(filas)} filas recibidas.")
    return pd.DataFrame(filas)

# MÉTODO B: cliente SDK oficial
def cargar_datos_sdk(tabla):
    if not _supabase_listo():
        return None
    print(f"📡 Conectando mediante SDK oficial → {tabla} ...")
    try:
        from supabase import create_client
        cliente = create_client(SUPABASE_URL, SUPABASE_KEY)
        filas, inicio, pagina = [], 0, 1000
        while True:
            lote = cliente.table(tabla).select("*").range(inicio, inicio + pagina - 1).execute().data
            filas += lote
            if len(lote) < pagina:
                break
            inicio += pagina
        return pd.DataFrame(filas)
    except Exception as e:
        print(f"❌ Error usando el SDK: {e}")
        return None

# ───────────── 3. GUARDAR datos ─────────────
def _a_registros(df):
    """DataFrame → lista de dicts lista para JSON (fechas ISO, NaN → null, enteros sin «.0»)."""
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_integer_dtype(df[c].dtype):            # incluye Int64 con nulos
            df[c] = pd.Series([None if pd.isna(v) else int(v) for v in df[c]], index=df.index, dtype=object)
        else:
            df[c] = df[c].map(lambda v: v.isoformat() if hasattr(v, "isoformat") and not pd.isna(v) else v)
    return json.loads(df.to_json(orient="records"))

def guardar_datos_api_rest(df, tabla, upsert_on=None, lote=500):
    """INSERT (o UPSERT si das `upsert_on`, p. ej. "id_evento" o "zona_codigo,periodo") por lotes."""
    if not _supabase_listo():
        return False
    registros = _a_registros(df)
    prefer = "return=minimal" + (",resolution=merge-duplicates" if upsert_on else "")
    for i in range(0, len(registros), lote):
        r = requests.post(f"{SUPABASE_URL}/rest/v1/{tabla}", params={"on_conflict": upsert_on} if upsert_on else None,
                          headers=_headers({"Prefer": prefer}), data=json.dumps(registros[i:i + lote]), timeout=60)
        if r.status_code not in (200, 201, 204):
            print(f"❌ Error guardando en «{tabla}» (lote {i // lote + 1}): {r.status_code}\n{r.text}")
            return False
    print(f"✅ {len(registros)} filas guardadas en «{tabla}»" + (f" (upsert por {upsert_on})." if upsert_on else "."))
    return True

def guardar_datos_sdk(df, tabla, upsert_on=None):
    if not _supabase_listo():
        return False
    try:
        from supabase import create_client
        cliente = create_client(SUPABASE_URL, SUPABASE_KEY)
        t = cliente.table(tabla)
        (t.upsert(_a_registros(df), on_conflict=upsert_on) if upsert_on else t.insert(_a_registros(df))).execute()
        print(f"✅ {len(df)} filas guardadas en «{tabla}» (SDK).")
        return True
    except Exception as e:
        print(f"❌ Error usando el SDK: {e}")
        return False

print("Funciones de Supabase listas ✔  ·  estado:", "ACTIVO" if USAR_SUPABASE else "desactivado (USAR_SUPABASE = False)")

## Paso 1 · Problema, artefacto y criterios de aceptación de la v0.5

**Artefacto:** paquete `georisk_twin` (instalable) + API FastAPI + demo + guía técnica.

**Los cuatro criterios de aceptación** (Fase de Evaluación del anteproyecto). Si alguno falla, la iteración v0.5 **no se cierra** y se vuelve a *Diseño y desarrollo*:

| # | Criterio | Cómo se verificará en este notebook |
|---|---|---|
| C1 | El paquete es **instalable** | Se compila un *wheel* y se instala en un directorio limpio (Paso 6) |
| C2 | La API tiene **documentación completa** | Todas las operaciones de la especificación OpenAPI tienen resumen, etiqueta y esquema de respuesta (Paso 8) |
| C3 | La demo es **reproducible por terceros** | Dos ejecuciones independientes producen la misma “huella” (Paso 9) |
| C4 | Hay **conexión mínima** de datos, indicadores y un módulo analítico | Los 3 módulos figuran como integrados y el flujo completo responde (Pasos 8 y 9) |

**Métricas complementarias** (matriz de operacionalización): cobertura de pruebas, tiempo de respuesta de los endpoints y número de módulos integrados.

In [ ]:
COBERTURA_MIN_PCT = 80          # cobertura mínima de pruebas unitarias (ajustable)
LATENCIA_REF_MS   = 500         # umbral de REFERENCIA para el tiempo de respuesta (el anteproyecto solo pide medirlo)
MODULOS_MINIMOS   = {"datos", "indicadores", "ia"}   # umbral mínimo de la v0.5

## Paso 2 · Estructura del paquete (Objetivo específico 1)

Se usa el *layout* `src/`, la forma recomendada para paquetes instalables porque evita importar por accidente el código de la carpeta de trabajo en lugar del instalado.

```
georisk_twin_repo/
├── pyproject.toml            ← empaquetado y dependencias
├── README.md                 ← guía técnica
├── CONTRIBUTING.md           ← plantilla de contribución
├── src/georisk_twin/
│   ├── __init__.py
│   ├── schemas.py            ← CONTRATOS de datos (Pydantic)
│   ├── api.py                ← endpoints FastAPI
│   └── modulos/              ← datos · indicadores · ia · (grafos, simulación, visualización pendientes)
├── tests/                    ← pytest
├── examples/                 ← demo reproducible
└── .github/workflows/ci.yml  ← integración continua
```

In [ ]:
if REPO.exists():
    shutil.rmtree(REPO)                                   # partimos de cero en cada ejecución
for carpeta in [PKG / "modulos", REPO / "tests", REPO / "examples", REPO / "docs", REPO / ".github" / "workflows"]:
    carpeta.mkdir(parents=True, exist_ok=True)
print("Carpetas creadas ✔")

In [ ]:
%%writefile georisk_twin_repo/pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "georisk-twin"
version = "0.5.0"
description = "Librería y API para operacionalizar un gemelo digital multimodal urbano-industrial (riesgo)."
readme = "README.md"
requires-python = ">=3.10"
dependencies = [
    "fastapi>=0.110",
    "pydantic>=2.5",
    "uvicorn>=0.27",
    "pandas>=2.0",
    "numpy>=1.24",
    "scikit-learn>=1.3",
]

[project.optional-dependencies]
dev = ["pytest>=8", "pytest-cov>=5", "httpx>=0.27", "requests>=2.31"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-q"

In [ ]:
%%writefile georisk_twin_repo/src/georisk_twin/__init__.py
"""georisk_twin · capa común de integración del gemelo digital multimodal urbano-industrial."""

__version__ = "0.5.0"

from .schemas import Incidente, IndicadorZona, PrioridadZona, EstadoModulo  # noqa: F401

## Paso 3 · Contratos de datos con Pydantic (API First Design)

Este es el corazón del enfoque **API First**: antes de programar la lógica se acuerda **qué datos se intercambian**. Cada equipo de tesis puede desarrollar contra estos contratos sin esperar a que el paquete esté terminado.

| Contrato | Para qué sirve |
|---|---|
| `Incidente` | Evento de seguridad que entrega el módulo de datos |
| `IndicadorZona` | Indicadores de riesgo calculados por zona |
| `PrioridadZona` | Resultado del módulo analítico (prioridad por zona) |
| `EstadoModulo` | Estado de integración de cada módulo del gemelo |
| `SolicitudIndicadores`, `ResultadoValidacion`, `PipelineOut` | Cuerpos de petición/respuesta de la API |

In [ ]:
%%writefile georisk_twin_repo/src/georisk_twin/schemas.py
"""Contratos de datos (Pydantic) de georisk_twin.

Son la "fuente de verdad" del enfoque API First: la API, la documentación OpenAPI y
las validaciones se generan a partir de estos modelos.
"""
from datetime import date
from typing import Literal, Optional

from pydantic import BaseModel, Field

Nivel = Literal["bajo", "medio", "alto", "critico"]
Prioridad = Literal["baja", "media", "alta"]
ModoModulo = Literal["real", "mock", "pendiente"]


class Incidente(BaseModel):
    """Evento de seguridad (contrato esperado del módulo de datos)."""
    id_evento: str = Field(min_length=1, description="Identificador único del evento", examples=["INC-0001"])
    fecha: date = Field(description="Fecha del evento (AAAA-MM-DD)")
    zona: str = Field(min_length=1, description="Código de zona", examples=["Z01"])
    tipo_evento: str = Field(min_length=1, examples=["Caída"])
    severidad: int = Field(ge=1, le=4, description="1 leve · 2 moderada · 3 grave · 4 crítica")
    dias_perdidos: int = Field(default=0, ge=0, description="Días de trabajo perdidos")


class IndicadorZona(BaseModel):
    """Indicadores de riesgo por zona (contrato del módulo de indicadores)."""
    zona: str
    n_incidentes: int = Field(ge=0)
    indice_frecuencia: float = Field(ge=0, description="Incidentes por 200 000 horas de exposición")
    indice_gravedad: float = Field(ge=0, description="Días perdidos por 200 000 horas de exposición")
    exposicion_horas: float = Field(gt=0)
    puntaje_riesgo: float = Field(ge=0, le=100, description="Puntaje compuesto 0-100")
    nivel: Nivel


class PrioridadZona(BaseModel):
    """Resultado del módulo analítico (IA): prioridad de intervención por zona."""
    zona: str
    cluster: int = Field(ge=0)
    prioridad: Prioridad
    puntaje_riesgo: float = Field(ge=0, le=100)


class EstadoModulo(BaseModel):
    """Estado de integración de un módulo del gemelo digital."""
    nombre: str
    origen_tesis: str
    modo: ModoModulo
    integrado: bool
    descripcion: str


class SolicitudIndicadores(BaseModel):
    incidentes: list[Incidente] = Field(min_length=1)
    horas_exposicion: dict[str, float] = Field(description="Horas de exposición por código de zona")


class ErrorValidacion(BaseModel):
    indice: int
    detalle: str


class ResultadoValidacion(BaseModel):
    total: int
    validos: int
    invalidos: int
    errores: list[ErrorValidacion]


class PipelineOut(BaseModel):
    """Resultado del flujo completo datos → indicadores → analítico."""
    semilla: int
    n_incidentes: int
    indicadores: list[IndicadorZona]
    prioridades: list[PrioridadZona]
    modulos_usados: list[str]

In [ ]:
# Los contratos ya se pueden ver (¡sin haber escrito aún ninguna lógica!). Esto es API First.
sys.path.insert(0, str(REPO / "src"))
import importlib
import georisk_twin.schemas as schemas
importlib.reload(schemas)

print("Contrato 'Incidente' → esquema JSON (extracto):")
print(json.dumps({k: v for k, v in schemas.Incidente.model_json_schema()["properties"].items()}, indent=2, ensure_ascii=False)[:900], "…")

# Ejemplo de validación automática del contrato
for caso in [{"id_evento": "INC-1", "fecha": "2026-03-10", "zona": "Z01", "tipo_evento": "Caída", "severidad": 2},
             {"id_evento": "INC-2", "fecha": "2026-03-10", "zona": "Z01", "tipo_evento": "Caída", "severidad": 9}]:
    try:
        schemas.Incidente.model_validate(caso); print("✔ válido  :", caso["id_evento"])
    except Exception as e:
        print("✘ rechazado:", caso["id_evento"], "→", e.errors()[0]["loc"], e.errors()[0]["msg"])

## Paso 4 · Módulos del gemelo digital y registro de módulos

Cada módulo implementa la misma interfaz mínima (`ModuloGemelo`) y declara su **modo**:

| Modo | Significado |
|---|---|
| `real` | Implementación funcional |
| `mock` | Simulado, pero **respeta el contrato**; se reemplaza cuando la otra tesis entregue su módulo |
| `pendiente` | Aún no integrado (no cuenta para el umbral mínimo) |

Un **registro** central permite que una tesis conecte su módulo con `registrar_modulo(...)` **sin modificar el paquete** (se demuestra en el Paso 10).

> 💡 **Gestión de bloqueos con mocks (Fase 4 de Kanban):** mientras SI-01/SI-02 no entregan, los módulos `datos` e `indicadores` funcionan con datos simulados y el resto del sistema ya puede probarse.

In [ ]:
%%writefile georisk_twin_repo/src/georisk_twin/modulos/base.py
"""Interfaz común que deben cumplir todos los módulos del gemelo digital."""
from abc import ABC

from ..schemas import EstadoModulo


class ModuloGemelo(ABC):
    """Clase base. Cada tesis hereda de aquí y completa sus atributos."""
    nombre: str = ""             # ranura: datos | indicadores | grafos | ia | simulacion | visualizacion
    origen_tesis: str = ""
    modo: str = "pendiente"      # real | mock | pendiente
    descripcion: str = ""

    def estado(self) -> EstadoModulo:
        return EstadoModulo(
            nombre=self.nombre,
            origen_tesis=self.origen_tesis,
            modo=self.modo,
            integrado=self.modo != "pendiente",
            descripcion=self.descripcion,
        )

In [ ]:
%%writefile georisk_twin_repo/src/georisk_twin/modulos/datos.py
"""Módulo de datos: ingesta y validación de incidentes (mock del subproducto SI-02)."""
from datetime import date, timedelta

import numpy as np
from pydantic import ValidationError

from ..schemas import ErrorValidacion, Incidente, ResultadoValidacion
from .base import ModuloGemelo


class ModuloDatos(ModuloGemelo):
    nombre = "datos"
    origen_tesis = "SI-02 (esperado)"
    modo = "mock"
    descripcion = "Ingesta y validación de incidentes. En v0.5 genera datos simulados que respetan el contrato Incidente."

    ZONAS = ("Z01", "Z02", "Z03", "Z04", "Z05", "Z06")
    TIPOS = ("Caída", "Golpe", "Quemadura", "Exposición química", "Atrapamiento", "Corte")
    PESOS_ZONA = (0.25, 0.20, 0.10, 0.20, 0.10, 0.15)

    def generar_mock(self, n: int = 200, seed: int = 42) -> list[Incidente]:
        """Genera n incidentes simulados de forma reproducible."""
        if n < 1:
            raise ValueError("n debe ser mayor o igual a 1")
        rng = np.random.default_rng(seed)
        inicio = date(2026, 1, 1)
        salida = []
        for i in range(1, n + 1):
            sev = int(rng.choice([1, 2, 3, 4], p=[0.45, 0.30, 0.18, 0.07]))
            dias = {1: 0, 2: int(rng.integers(1, 4)), 3: int(rng.integers(4, 16)), 4: int(rng.integers(16, 61))}[sev]
            salida.append(Incidente(
                id_evento=f"INC-{i:04d}",
                fecha=inicio + timedelta(days=int(rng.integers(0, 240))),
                zona=str(rng.choice(self.ZONAS, p=self.PESOS_ZONA)),
                tipo_evento=str(rng.choice(self.TIPOS)),
                severidad=sev,
                dias_perdidos=dias,
            ))
        return salida

    def validar(self, registros: list[dict]) -> tuple[list[Incidente], ResultadoValidacion]:
        """Valida registros crudos contra el contrato; informa cuáles fallan y por qué."""
        validos, errores = [], []
        for i, r in enumerate(registros):
            try:
                validos.append(Incidente.model_validate(r))
            except ValidationError as e:
                detalle = "; ".join(f"{'.'.join(map(str, x['loc']))}: {x['msg']}" for x in e.errors())
                errores.append(ErrorValidacion(indice=i, detalle=detalle))
        return validos, ResultadoValidacion(
            total=len(registros), validos=len(validos), invalidos=len(errores), errores=errores)

In [ ]:
%%writefile georisk_twin_repo/src/georisk_twin/modulos/indicadores.py
"""Módulo de indicadores: frecuencia, gravedad y puntaje de riesgo por zona (mock de SI-01)."""
import pandas as pd

from ..schemas import Incidente, IndicadorZona
from .base import ModuloGemelo


def _minmax(serie: pd.Series) -> pd.Series:
    rango = serie.max() - serie.min()
    return (serie - serie.min()) / rango if rango > 0 else serie * 0.0


def _nivel(puntaje: float) -> str:
    if puntaje < 25:
        return "bajo"
    if puntaje < 50:
        return "medio"
    if puntaje < 75:
        return "alto"
    return "critico"


class ModuloIndicadores(ModuloGemelo):
    nombre = "indicadores"
    origen_tesis = "SI-01 (esperado)"
    modo = "mock"
    descripcion = "Calcula índices de frecuencia y gravedad y un puntaje compuesto 0-100 por zona."

    PESOS = {"frecuencia": 0.4, "gravedad": 0.4, "exposicion": 0.2}
    BASE_HORAS = 200_000          # convención tipo OSHA: tasas por 200 000 horas trabajadas

    def calcular(self, incidentes: list[Incidente], horas_exposicion: dict[str, float]) -> list[IndicadorZona]:
        if not incidentes:
            raise ValueError("Se requiere al menos un incidente")
        df = pd.DataFrame([i.model_dump() for i in incidentes])
        por_zona = df.groupby("zona").agg(n=("id_evento", "count"), dias=("dias_perdidos", "sum")).reset_index()

        faltantes = sorted(set(por_zona["zona"]) - set(horas_exposicion))
        if faltantes:
            raise ValueError(f"Falta la exposición (horas) de las zonas: {', '.join(faltantes)}")
        por_zona["horas"] = por_zona["zona"].map(horas_exposicion)
        if (por_zona["horas"] <= 0).any():
            raise ValueError("Las horas de exposición deben ser mayores que cero")

        por_zona["frec"] = por_zona["n"] * self.BASE_HORAS / por_zona["horas"]
        por_zona["grav"] = por_zona["dias"] * self.BASE_HORAS / por_zona["horas"]
        por_zona["puntaje"] = 100 * (
            self.PESOS["frecuencia"] * _minmax(por_zona["frec"])
            + self.PESOS["gravedad"] * _minmax(por_zona["grav"])
            + self.PESOS["exposicion"] * _minmax(por_zona["horas"])
        )
        return [
            IndicadorZona(zona=r.zona, n_incidentes=int(r.n), indice_frecuencia=round(float(r.frec), 3),
                          indice_gravedad=round(float(r.grav), 3), exposicion_horas=float(r.horas),
                          puntaje_riesgo=round(float(r.puntaje), 2), nivel=_nivel(float(r.puntaje)))
            for r in por_zona.sort_values("zona").itertuples()
        ]

In [ ]:
%%writefile georisk_twin_repo/src/georisk_twin/modulos/analitico.py
"""Módulo analítico (IA): agrupa las zonas por perfil de riesgo y asigna prioridad de intervención."""
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from ..schemas import IndicadorZona, PrioridadZona
from .base import ModuloGemelo

_ETIQUETAS = {1: ["media"], 2: ["baja", "alta"], 3: ["baja", "media", "alta"]}


class ModuloAnalitico(ModuloGemelo):
    nombre = "ia"
    origen_tesis = "por definir (IND-SIS-01 / URB-01…URB-05)"
    modo = "real"
    descripcion = "Clustering K-Means sobre los indicadores para priorizar zonas (baja / media / alta)."

    def priorizar(self, indicadores: list[IndicadorZona], n_clusters: int = 3, seed: int = 0) -> list[PrioridadZona]:
        if not indicadores:
            raise ValueError("Se requiere al menos un indicador")
        k = max(1, min(n_clusters, 3, len(indicadores)))
        X = np.array([[i.indice_frecuencia, i.indice_gravedad, i.exposicion_horas, i.puntaje_riesgo] for i in indicadores])
        etiquetas = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(StandardScaler().fit_transform(X)) if k > 1 \
            else np.zeros(len(indicadores), dtype=int)

        # Ordenamos los clusters por puntaje medio para que "alta" sea siempre el de mayor riesgo
        puntaje_medio = {c: np.mean([i.puntaje_riesgo for i, e in zip(indicadores, etiquetas) if e == c]) for c in set(etiquetas)}
        orden = sorted(puntaje_medio, key=puntaje_medio.get)
        nombre_prioridad = {c: _ETIQUETAS[k][pos] for pos, c in enumerate(orden)}
        return [PrioridadZona(zona=i.zona, cluster=int(e), prioridad=nombre_prioridad[e], puntaje_riesgo=i.puntaje_riesgo)
                for i, e in zip(indicadores, etiquetas)]

In [ ]:
%%writefile georisk_twin_repo/src/georisk_twin/modulos/pendientes.py
"""Módulos aún no integrados en la v0.5. Ocupan su ranura para que el registro esté completo."""
from .base import ModuloGemelo


class ModuloGrafos(ModuloGemelo):
    nombre = "grafos"
    origen_tesis = "por definir (URB-01…URB-05)"
    modo = "pendiente"
    descripcion = "Grafos de infraestructura y propagación de riesgo (pendiente de integración)."


class ModuloSimulacion(ModuloGemelo):
    nombre = "simulacion"
    origen_tesis = "por definir (URB-01…URB-05)"
    modo = "pendiente"
    descripcion = "Simulación de escenarios (pendiente de integración)."


class ModuloVisualizacion(ModuloGemelo):
    nombre = "visualizacion"
    origen_tesis = "por definir (IND-SIS-01 / URB-01…URB-05)"
    modo = "pendiente"
    descripcion = "Visualización / dashboard (pendiente de integración)."

In [ ]:
%%writefile georisk_twin_repo/src/georisk_twin/modulos/__init__.py
"""Registro de módulos del gemelo digital: un único lugar donde se conectan los subproductos."""
from .analitico import ModuloAnalitico
from .base import ModuloGemelo
from .datos import ModuloDatos
from .indicadores import ModuloIndicadores
from .pendientes import ModuloGrafos, ModuloSimulacion, ModuloVisualizacion

SLOTS = ("datos", "indicadores", "grafos", "ia", "simulacion", "visualizacion")
_REGISTRO: dict[str, ModuloGemelo] = {}


def restablecer_registro() -> None:
    """Deja el registro en su estado inicial de la v0.5."""
    _REGISTRO.clear()
    for m in (ModuloDatos(), ModuloIndicadores(), ModuloGrafos(), ModuloAnalitico(),
              ModuloSimulacion(), ModuloVisualizacion()):
        _REGISTRO[m.nombre] = m


def registrar_modulo(modulo: ModuloGemelo) -> None:
    """Conecta (o reemplaza) el módulo de una ranura sin tocar el resto del paquete."""
    if modulo.nombre not in SLOTS:
        raise ValueError(f"Ranura desconocida '{modulo.nombre}'. Ranuras válidas: {', '.join(SLOTS)}")
    _REGISTRO[modulo.nombre] = modulo


def obtener_modulos() -> dict[str, ModuloGemelo]:
    return {s: _REGISTRO[s] for s in SLOTS}


restablecer_registro()

## Paso 5 · API con FastAPI (Objetivo específico 2)

Cada endpoint tiene `summary`, `tags` y un `response_model`; así **la especificación OpenAPI nace del código** y nunca queda desactualizada.

| Endpoint | Función |
|---|---|
| `GET /health` | Estado del servicio y versión |
| `GET /modulos` | Estado de integración de los 6 módulos |
| `GET /datos/mock` | Incidentes simulados reproducibles (`n`, `seed`) |
| `POST /datos/validar` | Valida registros crudos contra el contrato `Incidente` |
| `POST /indicadores/calcular` | Calcula indicadores por zona |
| `POST /analitico/priorizar` | Prioriza zonas con K-Means |
| `GET /demo/pipeline` | Ejecuta el flujo completo datos → indicadores → analítico |

In [ ]:
%%writefile georisk_twin_repo/src/georisk_twin/api.py
"""API REST de georisk_twin (FastAPI). La documentación OpenAPI se genera automáticamente."""
from fastapi import Body, FastAPI, HTTPException, Query

from . import __version__
from .modulos import obtener_modulos
from .schemas import (EstadoModulo, Incidente, IndicadorZona, PipelineOut, PrioridadZona,
                      ResultadoValidacion, SolicitudIndicadores)


def create_app() -> FastAPI:
    app = FastAPI(
        title="georisk_twin · API de integración del gemelo digital multimodal",
        version=__version__,
        description="Capa común que conecta los módulos de datos, indicadores, grafos, IA, simulación y "
                    "visualización del gemelo digital multimodal urbano-industrial.",
    )

    @app.get("/health", tags=["Sistema"], summary="Estado del servicio")
    def health() -> dict:
        return {"status": "ok", "version": __version__}

    @app.get("/modulos", response_model=list[EstadoModulo], tags=["Sistema"],
             summary="Estado de integración de los módulos")
    def modulos():
        return [m.estado() for m in obtener_modulos().values()]

    @app.get("/datos/mock", response_model=list[Incidente], tags=["Datos"],
             summary="Incidentes simulados (reproducibles)")
    def datos_mock(n: int = Query(200, ge=1, le=5000, description="Cantidad de incidentes"),
                   seed: int = Query(42, description="Semilla aleatoria")):
        return obtener_modulos()["datos"].generar_mock(n=n, seed=seed)

    @app.post("/datos/validar", response_model=ResultadoValidacion, tags=["Datos"],
              summary="Validar registros contra el contrato Incidente")
    def datos_validar(registros: list[dict] = Body(..., description="Lista de registros crudos")):
        _, resultado = obtener_modulos()["datos"].validar(registros)
        return resultado

    @app.post("/indicadores/calcular", response_model=list[IndicadorZona], tags=["Indicadores"],
              summary="Calcular indicadores de riesgo por zona")
    def indicadores_calcular(solicitud: SolicitudIndicadores):
        try:
            return obtener_modulos()["indicadores"].calcular(solicitud.incidentes, solicitud.horas_exposicion)
        except ValueError as e:
            raise HTTPException(status_code=422, detail=str(e))

    @app.post("/analitico/priorizar", response_model=list[PrioridadZona], tags=["Analítico (IA)"],
              summary="Priorizar zonas mediante clustering")
    def analitico_priorizar(indicadores: list[IndicadorZona]):
        try:
            return obtener_modulos()["ia"].priorizar(indicadores)
        except ValueError as e:
            raise HTTPException(status_code=422, detail=str(e))

    @app.get("/demo/pipeline", response_model=PipelineOut, tags=["Demo"],
             summary="Flujo completo datos → indicadores → analítico")
    def demo_pipeline(n: int = Query(300, ge=10, le=5000), seed: int = Query(42)):
        mods = obtener_modulos()
        incidentes = mods["datos"].generar_mock(n=n, seed=seed)
        zonas = sorted({i.zona for i in incidentes})
        horas = {z: 2000.0 + 400.0 * k for k, z in enumerate(zonas)}      # exposición simulada y determinista
        indicadores = mods["indicadores"].calcular(incidentes, horas)
        prioridades = mods["ia"].priorizar(indicadores)
        return PipelineOut(semilla=seed, n_incidentes=len(incidentes), indicadores=indicadores,
                           prioridades=prioridades, modulos_usados=["datos", "indicadores", "ia"])

    return app


app = create_app()

## Paso 6 · Instalación del paquete (Criterio C1 · HU-01)

Se hacen **dos** instalaciones con propósitos distintos:

1. **Editable** (`pip install -e`): la que usarán los desarrolladores del equipo; los cambios en el código se reflejan al instante.
2. **Prueba de instalación limpia:** se compila un *wheel* (el archivo que se publicaría) y se instala en una carpeta vacía; luego se importa **desde otra ruta** para comprobar que realmente funciona fuera del repositorio.

In [ ]:
# 1) Instalación editable (modo desarrollo)
code_pip, _ = ejecutar([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[dev]"], mostrar=True)
print("pip install -e →", "OK ✔" if code_pip == 0 else "FALLÓ ✘")

# En un kernel ya iniciado, aseguramos que el paquete sea importable
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
import importlib, georisk_twin
importlib.reload(georisk_twin)
print("Versión importada:", georisk_twin.__version__)

In [ ]:
# 2) Instalación limpia desde un wheel
DIST, LIMPIO = REPO / "dist", Path("/tmp/georisk_install_limpio")
for d in (DIST, LIMPIO):
    shutil.rmtree(d, ignore_errors=True)

code_build, _ = ejecutar([sys.executable, "-m", "build", "--wheel", "--outdir", str(DIST), str(REPO)], mostrar=False)
wheels = sorted(DIST.glob("*.whl"))
print("Wheel generado:", wheels[0].name if wheels else "— ninguno —", f"({wheels[0].stat().st_size/1024:.1f} KB)" if wheels else "")

inst_ok, detalle_inst = False, ""
if wheels:
    code_inst, _ = ejecutar([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", str(LIMPIO), str(wheels[0])], mostrar=False)
    env = {**os.environ, "PYTHONPATH": str(LIMPIO)}
    code_imp, salida = ejecutar(
        [sys.executable, "-c",
         "import georisk_twin as g; from georisk_twin.api import create_app; "
         "print(g.__version__); print(g.__file__)"],
        cwd="/tmp", env=env, mostrar=False)
    lineas = salida.splitlines()
    inst_ok = (code_inst == 0 and code_imp == 0 and len(lineas) >= 2
               and lineas[0] == "0.5.0" and lineas[1].startswith(str(LIMPIO)))
    detalle_inst = f"versión {lineas[0]} importada desde {lineas[1]}" if len(lineas) >= 2 else salida[-300:]

print("Instalación limpia + import desde otra ruta:", "OK ✔" if inst_ok else "FALLÓ ✘")
print("  ", detalle_inst)

## Paso 7 · Pruebas unitarias y cobertura (HU-04)

Se escribe la suite con `pytest` (las pruebas están en archivos reales dentro del repositorio, no en el notebook). Se prueban tres niveles: **contratos**, **módulos** y **API**. Después se mide la cobertura con `pytest-cov`.

In [ ]:
%%writefile georisk_twin_repo/tests/test_schemas.py
import pytest
from pydantic import ValidationError

from georisk_twin.schemas import Incidente, IndicadorZona

BASE = {"id_evento": "INC-1", "fecha": "2026-03-10", "zona": "Z01", "tipo_evento": "Caída", "severidad": 2}


def test_incidente_valido_usa_dias_perdidos_por_defecto():
    assert Incidente.model_validate(BASE).dias_perdidos == 0


@pytest.mark.parametrize("campo,valor", [("severidad", 9), ("severidad", 0), ("dias_perdidos", -1), ("zona", ""), ("fecha", "no-es-fecha")])
def test_incidente_rechaza_valores_invalidos(campo, valor):
    with pytest.raises(ValidationError):
        Incidente.model_validate({**BASE, campo: valor})


def test_indicador_exige_exposicion_positiva():
    with pytest.raises(ValidationError):
        IndicadorZona(zona="Z01", n_incidentes=1, indice_frecuencia=1, indice_gravedad=1,
                      exposicion_horas=0, puntaje_riesgo=10, nivel="bajo")

In [ ]:
%%writefile georisk_twin_repo/tests/test_modulos.py
import pytest

from georisk_twin.modulos import (SLOTS, ModuloAnalitico, ModuloDatos, ModuloGemelo, ModuloIndicadores,
                                  obtener_modulos, registrar_modulo, restablecer_registro)


@pytest.fixture(autouse=True)
def registro_limpio():
    restablecer_registro()
    yield
    restablecer_registro()


def _indicadores(n=200, seed=1):
    datos = ModuloDatos().generar_mock(n=n, seed=seed)
    zonas = sorted({d.zona for d in datos})
    return ModuloIndicadores().calcular(datos, {z: 2000.0 + 500 * i for i, z in enumerate(zonas)})


def test_mock_es_reproducible():
    assert ModuloDatos().generar_mock(50, seed=7) == ModuloDatos().generar_mock(50, seed=7)


def test_mock_rechaza_n_invalido():
    with pytest.raises(ValueError):
        ModuloDatos().generar_mock(0)


def test_validar_separa_validos_e_invalidos():
    crudos = [{"id_evento": "A", "fecha": "2026-01-01", "zona": "Z01", "tipo_evento": "Golpe", "severidad": 1},
              {"id_evento": "B", "fecha": "2026-01-01", "zona": "Z01", "tipo_evento": "Golpe", "severidad": 7}]
    validos, res = ModuloDatos().validar(crudos)
    assert (res.total, res.validos, res.invalidos) == (2, 1, 1) and len(validos) == 1
    assert res.errores[0].indice == 1 and "severidad" in res.errores[0].detalle


def test_indicadores_en_rango_y_con_nivel_valido():
    for i in _indicadores():
        assert 0 <= i.puntaje_riesgo <= 100
        assert i.nivel in {"bajo", "medio", "alto", "critico"}


def test_indicadores_exigen_exposicion_de_todas_las_zonas():
    datos = ModuloDatos().generar_mock(30)
    with pytest.raises(ValueError, match="Falta la exposición"):
        ModuloIndicadores().calcular(datos, {"Z01": 1000.0})


def test_indicadores_rechazan_horas_no_positivas_y_lista_vacia():
    datos = ModuloDatos().generar_mock(30)
    with pytest.raises(ValueError):
        ModuloIndicadores().calcular(datos, {z: 0.0 for z in ModuloDatos.ZONAS})
    with pytest.raises(ValueError):
        ModuloIndicadores().calcular([], {})


def test_analitico_cubre_todas_las_zonas_y_ordena_prioridades():
    ind = _indicadores()
    res = ModuloAnalitico().priorizar(ind)
    assert {r.zona for r in res} == {i.zona for i in ind}
    medio = lambda p: sum(r.puntaje_riesgo for r in res if r.prioridad == p) / max(1, sum(r.prioridad == p for r in res))
    assert medio("alta") >= medio("baja")


def test_analitico_con_pocas_zonas():
    uno = _indicadores()[:1]
    assert ModuloAnalitico().priorizar(uno)[0].prioridad == "media"
    dos = _indicadores()[:2]
    assert {r.prioridad for r in ModuloAnalitico().priorizar(dos)} <= {"baja", "alta"}
    with pytest.raises(ValueError):
        ModuloAnalitico().priorizar([])


def test_registro_inicial_tiene_tres_modulos_integrados():
    mods = obtener_modulos()
    assert tuple(mods) == SLOTS
    assert {n for n, m in mods.items() if m.estado().integrado} == {"datos", "indicadores", "ia"}


def test_registrar_modulo_nuevo_y_rechazar_ranura_desconocida():
    class GrafosDemo(ModuloGemelo):
        nombre, origen_tesis, modo, descripcion = "grafos", "demo", "mock", "demo"

    class Raro(ModuloGemelo):
        nombre = "no_existe"

    registrar_modulo(GrafosDemo())
    assert obtener_modulos()["grafos"].estado().integrado is True
    with pytest.raises(ValueError):
        registrar_modulo(Raro())

In [ ]:
%%writefile georisk_twin_repo/tests/test_api.py
import pytest
from fastapi.testclient import TestClient

from georisk_twin.api import create_app

client = TestClient(create_app())
INC = {"id_evento": "INC-1", "fecha": "2026-03-10", "zona": "Z01", "tipo_evento": "Caída", "severidad": 2, "dias_perdidos": 1}


def test_health():
    r = client.get("/health")
    assert r.status_code == 200 and r.json()["status"] == "ok"


def test_modulos_seis_ranuras_tres_integradas():
    datos = client.get("/modulos").json()
    assert len(datos) == 6 and sum(m["integrado"] for m in datos) == 3


def test_datos_mock_y_parametros_invalidos():
    assert len(client.get("/datos/mock", params={"n": 25}).json()) == 25
    assert client.get("/datos/mock", params={"n": 0}).status_code == 422


def test_validar_reporta_errores():
    r = client.post("/datos/validar", json=[INC, {**INC, "severidad": 8}]).json()
    assert (r["validos"], r["invalidos"]) == (1, 1)


def test_calcular_indicadores_ok_y_error_422():
    ok = client.post("/indicadores/calcular", json={"incidentes": [INC], "horas_exposicion": {"Z01": 1500}})
    assert ok.status_code == 200 and ok.json()[0]["zona"] == "Z01"
    falta = client.post("/indicadores/calcular", json={"incidentes": [INC], "horas_exposicion": {}})
    assert falta.status_code == 422 and "exposición" in falta.json()["detail"]
    vacio = client.post("/indicadores/calcular", json={"incidentes": [], "horas_exposicion": {}})
    assert vacio.status_code == 422


def test_priorizar_ok_y_error_422():
    pipeline = client.get("/demo/pipeline").json()
    r = client.post("/analitico/priorizar", json=pipeline["indicadores"])
    assert r.status_code == 200 and len(r.json()) == len(pipeline["indicadores"])
    assert client.post("/analitico/priorizar", json=[]).status_code == 422


def test_pipeline_es_determinista():
    a = client.get("/demo/pipeline", params={"seed": 5}).json()
    b = client.get("/demo/pipeline", params={"seed": 5}).json()
    assert a == b and a["modulos_usados"] == ["datos", "indicadores", "ia"]


def test_openapi_documenta_todas_las_operaciones():
    spec = client.get("/openapi.json").json()
    ops = [(p, m, d) for p, ms in spec["paths"].items() for m, d in ms.items()]
    assert len(ops) >= 7
    for ruta, metodo, d in ops:
        assert d.get("summary") and d.get("tags"), f"{metodo.upper()} {ruta} sin resumen/etiqueta"
        assert "200" in d["responses"], f"{metodo.upper()} {ruta} sin respuesta 200 documentada"

In [ ]:
env_tests = {**os.environ, "COVERAGE_FILE": str(REPO / ".coverage")}
code_t, salida_t = ejecutar(
    [sys.executable, "-m", "pytest", "--cov=georisk_twin", "--cov-report=term-missing", "--color=no",
     f"--cov-report=json:{REPO / 'coverage.json'}", "-p", "no:cacheprovider"],
    cwd=REPO, env=env_tests, mostrar=True)

cov = json.loads((REPO / "coverage.json").read_text())
cov_pct = round(cov["totals"]["percent_covered"], 1)
tests_ok = code_t == 0
resumen_tests = [l for l in salida_t.splitlines() if " passed" in l or " failed" in l][-1:]
print(f"\nPruebas: {'TODAS PASARON ✔' if tests_ok else 'HAY FALLOS ✘'} · {resumen_tests[0].strip('= ') if resumen_tests else ''}")
print(f"Cobertura total: {cov_pct} %  (mínimo exigido: {COBERTURA_MIN_PCT} %) → {'✔' if cov_pct >= COBERTURA_MIN_PCT else '✘'}")

## Paso 8 · Servicio en ejecución, OpenAPI y tiempos de respuesta (Criterio C2 y C4)

Se levanta el servidor real (Uvicorn) en segundo plano y se hace lo que haría un cliente externo: leer `/modulos`, descargar la especificación OpenAPI y medir el **tiempo de respuesta** de cada endpoint.

In [ ]:
import uvicorn
from georisk_twin.api import app

PUERTO = 8010
URL = f"http://127.0.0.1:{PUERTO}"
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PUERTO, log_level="warning"))
hilo = threading.Thread(target=server.run, daemon=True); hilo.start()

for _ in range(50):
    try:
        if requests.get(f"{URL}/health", timeout=1).status_code == 200: break
    except requests.ConnectionError:
        time.sleep(0.2)

print("Servicio activo:", requests.get(f"{URL}/health").json())
estado_modulos = pd.DataFrame(requests.get(f"{URL}/modulos").json())[["nombre", "modo", "integrado", "origen_tesis"]]
estado_modulos

In [ ]:
spec = requests.get(f"{URL}/openapi.json").json()
(REPO / "docs" / "openapi.json").write_text(json.dumps(spec, ensure_ascii=False, indent=2))

filas = []
for ruta, metodos in spec["paths"].items():
    for metodo, d in metodos.items():
        filas.append({"método": metodo.upper(), "ruta": ruta, "etiqueta": d["tags"][0], "resumen": d["summary"],
                      "tiene_resp_200": "200" in d["responses"]})
tabla_api = pd.DataFrame(filas)
openapi_ok = bool(len(tabla_api) >= 7 and tabla_api["resumen"].notna().all() and tabla_api["tiene_resp_200"].all())
print(f"OpenAPI {spec['openapi']} · {len(tabla_api)} operaciones · documentación completa: {'✔' if openapi_ok else '✘'}")
tabla_api

In [ ]:
# Swagger embebido (solo en Colab; fuera de Colab abre {URL}/docs)
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PUERTO, path="/docs", height=620)
except ImportError:
    print(f"No estás en Colab: abre {URL}/docs en tu navegador.")

In [ ]:
# ---------- Benchmark de endpoints ----------
from georisk_twin.modulos import ModuloDatos, ModuloIndicadores, ModuloAnalitico
_datos = ModuloDatos().generar_mock(200, seed=1)
_horas = {z: 2000.0 + 400 * i for i, z in enumerate(sorted({d.zona for d in _datos}))}
_ind   = ModuloIndicadores().calcular(_datos, _horas)

payload_validar = [d.model_dump(mode="json") for d in _datos]
payload_calc    = {"incidentes": payload_validar, "horas_exposicion": _horas}
payload_prior   = [i.model_dump(mode="json") for i in _ind]

pruebas = [
    ("GET  /health",              lambda s: s.get(f"{URL}/health")),
    ("GET  /modulos",             lambda s: s.get(f"{URL}/modulos")),
    ("GET  /datos/mock (n=200)",  lambda s: s.get(f"{URL}/datos/mock", params={"n": 200})),
    ("POST /datos/validar (200)", lambda s: s.post(f"{URL}/datos/validar", json=payload_validar)),
    ("POST /indicadores/calcular",lambda s: s.post(f"{URL}/indicadores/calcular", json=payload_calc)),
    ("POST /analitico/priorizar", lambda s: s.post(f"{URL}/analitico/priorizar", json=payload_prior)),
    ("GET  /demo/pipeline",       lambda s: s.get(f"{URL}/demo/pipeline", params={"n": 300})),
]
REPS = 40
ses = requests.Session()
for _, f in pruebas: f(ses)                                   # calentamiento

med = []
for nombre, f in pruebas:
    for _ in range(REPS):
        t0 = time.perf_counter(); r = f(ses); ms = (time.perf_counter() - t0) * 1000
        med.append({"endpoint": nombre, "ms": ms, "ok": r.status_code == 200})
df_bench = pd.DataFrame(med)

resumen_bench = (df_bench.groupby("endpoint", sort=False)
                 .agg(exito_pct=("ok", lambda s: 100 * s.mean()), media_ms=("ms", "mean"),
                      mediana_ms=("ms", "median"), p95_ms=("ms", lambda s: s.quantile(.95)), max_ms=("ms", "max")).round(2))
resumen_bench

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
orden = list(resumen_bench.index)
ax.boxplot([df_bench[df_bench.endpoint == e].ms for e in orden], vert=False, showfliers=True)
ax.set_yticks(range(1, len(orden) + 1)); ax.set_yticklabels(orden)
ax.axvline(LATENCIA_REF_MS, color="crimson", ls="--", label=f"umbral de referencia {LATENCIA_REF_MS} ms")
ax.set_xscale("log"); ax.set_xlabel("tiempo de respuesta (ms, escala log)"); ax.invert_yaxis(); ax.legend()
ax.set_title("georisk_twin v0.5 · Tiempo de respuesta por endpoint")
plt.tight_layout(); plt.show()

p95_max = float(resumen_bench.p95_ms.max())
print(f"Peor p95 entre endpoints: {p95_max:.1f} ms → {'por debajo' if p95_max < LATENCIA_REF_MS else 'POR ENCIMA'} del umbral de referencia.")

## Paso 9 · Demo de integración reproducible (Criterio C3 y C4 · HU-06)

La demo es un **script independiente** (`examples/demo_integracion.py`) que cualquier persona con acceso al repositorio puede ejecutar. Hace el recorrido completo por la API:

`datos → validar → indicadores → analítico (IA)` y termina imprimiendo una **huella (hash)** del resultado.

**¿Cómo se demuestra la reproducibilidad?** Se ejecuta **dos veces en procesos distintos y desde otra carpeta**. Si la huella coincide, el resultado es idéntico.

In [ ]:
%%writefile georisk_twin_repo/examples/demo_integracion.py
"""Demo reproducible de georisk_twin v0.5.

Uso:  python examples/demo_integracion.py --seed 42 --n 300
Recorre datos → validación → indicadores → analítico (IA) y imprime una huella (hash) del resultado.
"""
import argparse
import hashlib
import json

from fastapi.testclient import TestClient

from georisk_twin import __version__
from georisk_twin.api import create_app


def ejecutar(seed: int, n: int) -> dict:
    client = TestClient(create_app())

    assert client.get("/health").json()["status"] == "ok"
    modulos = client.get("/modulos").json()

    incidentes = client.get("/datos/mock", params={"n": n, "seed": seed}).json()
    validacion = client.post("/datos/validar", json=incidentes).json()
    assert validacion["invalidos"] == 0, "El mock debe cumplir el contrato"

    zonas = sorted({i["zona"] for i in incidentes})
    horas = {z: 2000.0 + 400.0 * k for k, z in enumerate(zonas)}
    indicadores = client.post("/indicadores/calcular", json={"incidentes": incidentes, "horas_exposicion": horas}).json()
    prioridades = client.post("/analitico/priorizar", json=indicadores).json()

    return {
        "version": __version__,
        "semilla": seed,
        "modulos_integrados": [m["nombre"] for m in modulos if m["integrado"]],
        "validacion": validacion,
        "indicadores": indicadores,
        "prioridades": prioridades,
    }


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--n", type=int, default=300)
    args = ap.parse_args()

    res = ejecutar(args.seed, args.n)
    print(f"georisk_twin v{res['version']} · módulos integrados: {', '.join(res['modulos_integrados'])}")
    print(f"Incidentes validados: {res['validacion']['validos']}/{res['validacion']['total']}")
    print(f"{'zona':<6}{'puntaje':>9}  {'nivel':<8}{'prioridad':<10}")
    prio = {p["zona"]: p["prioridad"] for p in res["prioridades"]}
    for i in sorted(res["indicadores"], key=lambda x: -x["puntaje_riesgo"]):
        print(f"{i['zona']:<6}{i['puntaje_riesgo']:>9.2f}  {i['nivel']:<8}{prio[i['zona']]:<10}")
    huella = hashlib.sha256(json.dumps(res, sort_keys=True).encode()).hexdigest()[:16]
    print(f"HUELLA: {huella}")


if __name__ == "__main__":
    main()

In [ ]:
script = REPO / "examples" / "demo_integracion.py"
corridas = []
for k in (1, 2):
    c, salida = ejecutar([sys.executable, str(script), "--seed", "42", "--n", "300"], cwd="/tmp", mostrar=(k == 1))
    huella = next((l.split(":")[1].strip() for l in salida.splitlines() if l.startswith("HUELLA")), None)
    corridas.append((c, huella))

demo_ok = all(c == 0 for c, _ in corridas) and corridas[0][1] is not None and corridas[0][1] == corridas[1][1]
print(f"\nCorrida 1 → huella {corridas[0][1]}   |   Corrida 2 → huella {corridas[1][1]}")
print("Demo reproducible:", "SÍ ✔" if demo_ok else "NO ✘")

# Control negativo: con otra semilla la huella DEBE cambiar (si no cambiara, la huella no demostraría nada)
_, s3 = ejecutar([sys.executable, str(script), "--seed", "7"], cwd="/tmp", mostrar=False)
h3 = next((l.split(":")[1].strip() for l in s3.splitlines() if l.startswith("HUELLA")), None)
print(f"Control negativo (semilla 7) → huella {h3}  → {'distinta, como se esperaba ✔' if h3 != corridas[0][1] else 'IGUAL ✘'}")

## Paso 9B · (Opcional) Flujo con datos de Supabase: cargar → validar → calcular → priorizar → guardar

Es la demostración de que `georisk_twin` funciona con una **fuente externa real**: los registros se leen de Supabase, se **validan contra el contrato** `Incidente` mediante la API (los inválidos se informan, no se descartan en silencio), se calculan los indicadores, se prioriza con el módulo analítico y el resultado se **guarda** en Supabase.

- `COLUMNAS_MAP` sirve si tus columnas se llaman distinto (p. ej. `{"fecha_evento": "fecha", "zona_codigo": "zona"}`).
- `SEMBRAR_CON_MOCK = True` sube 300 incidentes simulados a tu tabla para poder probar el flujo completo.
- El servidor de la API del Paso 8 debe seguir activo (se detiene en el Paso 13).

In [ ]:
TABLA_ACCIDENTES = "accidentes_georisk"     # ← tu tabla de incidentes (p. ej. "nombreDetabla")
TABLA_RESULTADOS = "georisk_resultados"     # ← tabla donde se guardan los resultados
COLUMNAS_MAP     = {}                       # {"columna_en_tu_tabla": "columna_del_contrato"}
SEMBRAR_CON_MOCK = False                    # True → sube incidentes simulados a TABLA_ACCIDENTES

CONTRATO = ["id_evento", "fecha", "zona", "tipo_evento", "severidad", "dias_perdidos"]

if USAR_SUPABASE:
    if SEMBRAR_CON_MOCK:
        mock = pd.DataFrame(requests.get(f"{URL}/datos/mock", params={"n": 300, "seed": 42}).json())
        guardar_datos_api_rest(mock, TABLA_ACCIDENTES, upsert_on="id_evento")

    crudo = cargar_datos_api_rest(TABLA_ACCIDENTES)
    if crudo is not None and len(crudo):
        crudo = crudo.rename(columns=COLUMNAS_MAP)
        if "dias_perdidos" in crudo:
            crudo["dias_perdidos"] = crudo["dias_perdidos"].fillna(0)
        faltan = [c for c in CONTRATO if c not in crudo.columns]
        if faltan:
            print("⚠️ Faltan columnas del contrato:", faltan, "→ usa COLUMNAS_MAP.")
        else:
            registros = json.loads(crudo[CONTRATO].to_json(orient="records"))

            # 1) Validación contra el contrato (vía API)
            val = requests.post(f"{URL}/datos/validar", json=registros).json()
            print(f"Validación: {val['validos']} válidos · {val['invalidos']} inválidos de {val['total']}")
            for e in val["errores"][:5]:
                print(f"   fila {e['indice']}: {e['detalle']}")
            malos = {e["indice"] for e in val["errores"]}
            buenos = [r for i, r in enumerate(registros) if i not in malos]

            # 2) Indicadores y priorización (vía API)
            if buenos:
                zonas_ok = sorted({r["zona"] for r in buenos})
                horas = {z: 2000.0 + 400.0 * k for k, z in enumerate(zonas_ok)}   # exposición simulada: reemplazar por la real
                ind = requests.post(f"{URL}/indicadores/calcular", json={"incidentes": buenos, "horas_exposicion": horas}).json()
                pri = requests.post(f"{URL}/analitico/priorizar", json=ind).json()
                df_res = (pd.DataFrame(ind).merge(pd.DataFrame(pri)[["zona", "cluster", "prioridad"]], on="zona")
                          .assign(version=georisk_twin.__version__))
                display(df_res)

                # 3) Guardar resultados (upsert por zona)
                guardar_datos_api_rest(df_res, TABLA_RESULTADOS, upsert_on="zona")
else:
    print("Supabase desactivado: el flujo con datos externos no se ejecutó. (USAR_SUPABASE = True para usarlo.)")

## Paso 10 · Plantilla de contribución y conexión de un módulo nuevo (HU-05)

Para que otra tesis conecte su subproducto **sin romper la estructura**, el repositorio incluye una guía de contribución. Además se demuestra en vivo el mecanismo: se conecta un módulo de **grafos de ejemplo** con `registrar_modulo(...)` y la API lo reconoce **sin modificar ningún archivo del paquete**.

In [ ]:
%%writefile georisk_twin_repo/CONTRIBUTING.md
# Guía para conectar un subproducto a `georisk_twin`

Esta guía es para los equipos de las demás tesis (SI-01, SI-02, IND-SIS-01, URB-01 … URB-05).

## 1. Elige tu ranura
`datos` · `indicadores` · `grafos` · `ia` · `simulacion` · `visualizacion`

## 2. Respeta los contratos
Tus entradas y salidas deben cumplir los modelos de `georisk_twin/schemas.py`
(`Incidente`, `IndicadorZona`, `PrioridadZona`). **No necesitas cambiar tu implementación interna**:
solo adaptar la frontera (entrada/salida).

## 3. Crea tu módulo
```python
from georisk_twin.modulos import ModuloGemelo, registrar_modulo

class MiModulo(ModuloGemelo):
    nombre = "grafos"                    # una de las ranuras
    origen_tesis = "URB-0X"
    modo = "real"                        # "real" | "mock" | "pendiente"
    descripcion = "Qué hace mi módulo"

    def mi_funcion(self, entrada):       # tu lógica
        ...

registrar_modulo(MiModulo())
```

## 4. Pruebas obligatorias
- Añade al menos una prueba en `tests/` que valide tu frontera con el contrato.
- Ejecuta `pytest --cov=georisk_twin` (cobertura mínima del proyecto: 80 %).

## 5. Flujo de trabajo (Kanban)
1. Toma una tarjeta de **Por hacer** y muévela a **En progreso**.
2. Abre un *pull request*; pasa a **En revisión**.
3. Otra persona revisa el código.
4. Al fusionar, GitHub Actions ejecuta las pruebas. Solo si pasan, la tarjeta pasa a **Hecho**.

## 6. Si tu módulo aún no está listo
Usa `modo = "mock"` con datos simulados que respeten el contrato. Así el resto del equipo puede avanzar.

In [ ]:
%%writefile georisk_twin_repo/README.md
# georisk_twin

Librería y API en Python que operacionaliza la metodología de un **gemelo digital multimodal urbano-industrial** (riesgo).

## Instalación
```bash
pip install -e ".[dev]"
```

## Levantar la API
```bash
uvicorn georisk_twin.api:app --reload
# Documentación interactiva: http://127.0.0.1:8000/docs
```

## Demo reproducible
```bash
python examples/demo_integracion.py --seed 42
```

## Pruebas
```bash
pytest --cov=georisk_twin
```

## Módulos (v0.5)
| Ranura | Estado |
|---|---|
| datos | mock (contrato listo) |
| indicadores | mock (contrato listo) |
| ia | real (K-Means) |
| grafos · simulacion · visualizacion | pendientes |

Para conectar tu subproducto, lee [CONTRIBUTING.md](CONTRIBUTING.md).

In [ ]:
# ---- Demostración: un equipo ajeno conecta su módulo SIN tocar el paquete ----
from fastapi.testclient import TestClient
from georisk_twin.modulos import ModuloGemelo, registrar_modulo, restablecer_registro
from georisk_twin.api import create_app

class GrafosDeEjemplo(ModuloGemelo):                 # esto lo escribiría, por ejemplo, un equipo URB-0X
    nombre = "grafos"
    origen_tesis = "URB-0X (ejemplo)"
    modo = "mock"
    descripcion = "Grafo de ejemplo conectado por un equipo externo mediante registrar_modulo()."

cliente = TestClient(create_app())
antes = sum(m["integrado"] for m in cliente.get("/modulos").json())

registrar_modulo(GrafosDeEjemplo())
despues_json = cliente.get("/modulos").json()
despues = sum(m["integrado"] for m in despues_json)

contribucion_ok = (antes == 3 and despues == 4 and (REPO / "CONTRIBUTING.md").exists())
print(f"Módulos integrados: antes = {antes} → después = {despues}")
print(pd.DataFrame(despues_json)[["nombre", "modo", "integrado", "origen_tesis"]].to_string(index=False))

restablecer_registro()                               # volvemos al estado oficial v0.5 para la evaluación
print("\nRegistro restablecido →", sum(m["integrado"] for m in cliente.get("/modulos").json()), "módulos integrados")

## Paso 11 · Integración continua (GitHub Actions)

El anteproyecto plantea validar cada cambio automáticamente. Se genera el archivo de flujo de trabajo que, una vez subido el repositorio a GitHub, ejecutará **las mismas pruebas que acabamos de correr aquí** en tres versiones de Python y bloqueará los cambios que bajen la cobertura del mínimo.

> Colab no puede ejecutar GitHub Actions; el valor del piloto es dejar el archivo listo y comprobado en local (Paso 7).

In [ ]:
%%writefile georisk_twin_repo/.github/workflows/ci.yml
name: CI

on:
  push:
    branches: [main]
  pull_request:

jobs:
  pruebas:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - name: Instalar paquete y dependencias de desarrollo
        run: pip install -e ".[dev]"
      - name: Ejecutar pruebas con cobertura mínima del 80 %
        run: pytest --cov=georisk_twin --cov-fail-under=80
      - name: Demo reproducible
        run: python examples/demo_integracion.py --seed 42

## Paso 12 · Evaluación: criterios de aceptación, métricas y trazabilidad de historias de usuario

Todo lo que sigue se calcula **a partir de los resultados reales** de los pasos anteriores (nada está escrito a mano).

In [ ]:
# C4: conexión mínima → los 3 módulos mínimos integrados Y el flujo completo respondiendo
mods_estado = {m["nombre"]: m for m in requests.get(f"{URL}/modulos").json()}
integrados = {n for n, m in mods_estado.items() if m["integrado"]}
flujo = requests.get(f"{URL}/demo/pipeline", params={"n": 300}).json()
conexion_minima_ok = MODULOS_MINIMOS <= integrados and len(flujo["indicadores"]) > 0 and len(flujo["prioridades"]) > 0

criterios = pd.DataFrame([
    ["C1", "Paquete instalable",                               inst_ok,            detalle_inst],
    ["C2", "API con documentación completa (OpenAPI)",         openapi_ok,         f"{len(tabla_api)} operaciones con resumen, etiqueta y respuesta 200"],
    ["C3", "Demo reproducible por terceros",                   demo_ok,            f"huellas: {corridas[0][1]} = {corridas[1][1]}"],
    ["C4", "Conexión mínima (datos + indicadores + analítico)", conexion_minima_ok, f"integrados: {', '.join(sorted(integrados))}"],
], columns=["ID", "Criterio de aceptación", "Cumple", "Evidencia"])

metricas = pd.DataFrame([
    ["Cobertura de pruebas unitarias",          f"{cov_pct} %",                       cov_pct >= COBERTURA_MIN_PCT],
    ["Pruebas unitarias aprobadas",             "todas" if tests_ok else "con fallos", tests_ok],
    ["Peor p95 de tiempo de respuesta",         f"{p95_max:.1f} ms",                  p95_max < LATENCIA_REF_MS],
    ["Éxito de solicitudes en el benchmark",    f"{100*df_bench.ok.mean():.1f} %",    df_bench.ok.mean() == 1.0],
    ["Módulos integrados sobre el total",       f"{len(integrados)} de {len(mods_estado)}", MODULOS_MINIMOS <= integrados],
], columns=["Métrica", "Resultado", "Dentro del objetivo"])

fmt = lambda s: s.map({True: "✅ Sí", False: "❌ No"})
c_vis, m_vis = criterios.copy(), metricas.copy()
c_vis["Cumple"], m_vis["Dentro del objetivo"] = fmt(c_vis["Cumple"]), fmt(m_vis["Dentro del objetivo"])
display(c_vis); display(m_vis)

In [ ]:
# ---------- Trazabilidad: historias de usuario ↔ evidencia ----------
from pydantic import ValidationError
from georisk_twin.schemas import Incidente as _Inc

def _rechaza_dato_invalido():
    try:
        _Inc.model_validate({"id_evento": "x", "fecha": "2026-01-01", "zona": "Z01", "tipo_evento": "t", "severidad": 9})
        return False
    except ValidationError:
        return True

hu = pd.DataFrame([
    ["HU-01", "Instalar el paquete con pip",                              "Wheel compilado e instalado en carpeta limpia (C1)",                    inst_ok],
    ["HU-02", "Modelos Pydantic estandarizados para enviar datos",         "El contrato Incidente rechaza datos fuera de rango",                    _rechaza_dato_invalido()],
    ["HU-03", "Consultar la documentación OpenAPI de los endpoints",       "Especificación completa y guardada en docs/openapi.json (C2)",          openapi_ok and (REPO / "docs" / "openapi.json").exists()],
    ["HU-04", "Ejecutar pruebas unitarias automatizadas",                  f"pytest aprobado con {cov_pct} % de cobertura; flujo CI generado",      tests_ok and cov_pct >= COBERTURA_MIN_PCT and (REPO / ".github/workflows/ci.yml").exists()],
    ["HU-05", "Plantilla de contribución para conectar un módulo",         "CONTRIBUTING.md + módulo externo conectado sin tocar el paquete",       contribucion_ok],
    ["HU-06", "Ejecutar una demo reproducible",                            "Misma huella en dos corridas independientes (C3)",                      demo_ok],
], columns=["ID", "Historia de usuario (resumen)", "Evidencia", "Cumple"])
hu["Cumple"] = hu["Cumple"].map({True: "✅ Sí", False: "❌ No"})
hu

In [ ]:
# ---------- Veredicto de la iteración v0.5 (regla del anteproyecto) ----------
todos_ok = bool(criterios["Cumple"].all())
print("=" * 72)
if todos_ok:
    print("VEREDICTO: los 4 criterios de aceptación se cumplen → la iteración v0.5 SE PUEDE CERRAR ✅")
else:
    fallan = ", ".join(criterios.loc[~criterios["Cumple"], "ID"])
    print(f"VEREDICTO: incumple {fallan} → el artefacto NO está apto; se vuelve a Diseño y desarrollo ❌")
print("=" * 72)

## Paso 13 · Entregable, limitaciones y próximos pasos

Se detiene el servidor y se empaqueta todo el repositorio en un `.zip` listo para descargar y subir a GitHub.

**Qué quedó demostrado**
- La estructura común es **viable y liviana**: un paquete con contratos Pydantic, un registro de módulos y una API autodocumentada.
- Un módulo externo se conecta con **una sola llamada** (`registrar_modulo`), sin modificar el paquete.
- La evaluación es **automática y repetible**, con criterios explícitos y regla de cierre de iteración.

**Limitaciones (conviene declararlas en la tesis)**
- Datos e indicadores son **mocks**; deben reemplazarse cuando SI-01 y SI-02 entreguen sus subproductos. Las fórmulas de indicadores son ilustrativas.
- Los módulos de grafos, simulación y visualización **no están integrados** (3 de 6 = umbral mínimo de la v0.5).
- La prueba de reproducibilidad se hizo en un mismo sistema; para demostrarla “en distintos entornos” conviene repetirla en Windows/macOS/Linux (la matriz de Python del CI ayuda).
- El umbral de 500 ms es **de referencia**: el anteproyecto pide medir el tiempo, pero no fija un valor.

**Próximos pasos sugeridos**
1. Subir `georisk_twin_repo/` a GitHub y activar Actions y GitHub Pages para la documentación.
2. Crear el tablero Kanban en Trello (Backlog · Por hacer · En progreso · En revisión · Bloqueado · Hecho) con una tarjeta por historia de usuario.
3. Acordar con cada equipo el contrato definitivo y reemplazar los mocks.

In [ ]:
# Cierre ordenado y empaquetado del entregable
server.should_exit = True
hilo.join(timeout=5)

for basura in [REPO / ".coverage", REPO / "coverage.json", REPO / "build"]:
    if basura.is_file(): basura.unlink()
    elif basura.is_dir(): shutil.rmtree(basura, ignore_errors=True)
for egg in (REPO / "src").glob("*.egg-info"):
    shutil.rmtree(egg, ignore_errors=True)
for cache in REPO.rglob("__pycache__"):
    shutil.rmtree(cache, ignore_errors=True)

zip_path = shutil.make_archive("georisk_twin_v0.5_repo", "zip", root_dir=REPO.parent, base_dir=REPO.name)
print("Servicio detenido ✔")
print(f"Entregable: {zip_path}  ({Path(zip_path).stat().st_size/1024:.1f} KB)\n")

print("Contenido del repositorio:")
for p in sorted(REPO.rglob("*")):
    if p.is_file() and "dist" not in p.parts:
        print("  ", p.relative_to(REPO.parent))

# En Colab: descarga directa
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass